# Day 09: Cloud Database Connectivity & Neon PostgreSQL JDBC Lakehouse Ingestion
> **Part 9 of 10 in the PySpark Production Data Engineering Series**

[![PySpark](https://img.shields.io/badge/PySpark-3.5%2B%20%2F%204.0%2B-F26522?style=for-the-badge&logo=apachespark&logoColor=white)](https://spark.apache.org/)
[![Python](https://img.shields.io/badge/Python-3.10%2B-3776AB?style=for-the-badge&logo=python&logoColor=white)](https://python.org)
[![Status](https://img.shields.io/badge/Module-Day%2009%20of%2010-008080?style=for-the-badge)](../README.md)

---

## ☁️ Day 09: Cloud Database Connectivity & Neon PostgreSQL JDBC Lakehouse Ingestion
### 🎯 Learning Objectives:
1. Architect a **Hybrid Cloud Lakehouse**: Ingesting from operational cloud OLTP (Neon Serverless PostgreSQL) into a distributed analytics lake.
2. Configure Apache Spark with external PostgreSQL JDBC drivers (`postgresql-42.6.0.jar`).
3. Leverage dual ingestion patterns:
   - Direct JDBC driver streaming for large distributed tables.
   - `psycopg2` + Pandas to Spark DataFrame bridge for serverless environments.
4. Perform distributed relational joins and data cleansing on cloud-ingested datasets.
5. Export curated business data into local partitioned Parquet Gold lakehouse tables.
---


PostgreSQL
    │
    │ JDBC
    ▼
  PySpark
    │
    ├── Read
    ├── Transform
    ├── Filter
    └── Aggregate
    │
    ▼
Parquet


### ⚙️ Initialize SparkSession for Day 09


In [287]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Day9_Postgres_Pipeline")
    .master("local[*]")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark started with PostgreSQL JDBC support successfully!")

Spark started with PostgreSQL JDBC support successfully!


### ⚙️ Initialize SparkSession for Day 09


In [288]:
from pyspark.sql import SparkSession

# Initialize Spark Session with PostgreSQL JDBC driver package
spark = SparkSession.builder \
    .appName("Day9_Neon_Pipeline") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

print("Spark session successfully created with PostgreSQL JDBC driver package!")

Spark session successfully created with PostgreSQL JDBC driver package!


In [289]:
# Replace these with your cloud PostgreSQL credentials (e.g., from Neon or Supabase)
DB_HOST = "ep-square-frost-b3yvvjy4-pooler.c-4.ap-southeast-1.aws.neon.tech"  # e.g., ep-cool-cloud-123456.us-east-2.aws.neon.tech
DB_NAME = "neondb"                        # or de_project
DB_USER = "neondb_owner"                 # e.g., neondb_owner
DB_PASSWORD = "npg_yvpcVYCW8F2f"             # your database password

# Direct URL format using Neon's connection string parameters
jdbc_url = "jdbc:postgresql://ep-square-frost-b3yvvjy4-pooler.c-4.ap-southeast-1.aws.neon.tech/neondb?sslmode=require"

connection_properties = {
    "user": DB_USER,
    "password": DB_PASSWORD,
    "driver": "org.postgresql.Driver"
}

print("Configuration ready")

Configuration ready


!pip install psycopg2-binary


In [291]:
import psycopg2

# Connect to cloud PostgreSQL using psycopg2
conn = psycopg2.connect(
    host=DB_HOST,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    sslmode="require"
)
cursor = conn.cursor()

# 1. Create tables
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id INT PRIMARY KEY,
    customer_id INT,
    customer_name VARCHAR(100),
    city VARCHAR(100),
    product VARCHAR(100),
    quantity INT,
    price DECIMAL(10,2),
    order_date DATE
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INT PRIMARY KEY,
    customer_name VARCHAR(100),
    city VARCHAR(100),
    signup_date DATE
);
""")

# 2. Insert Orders Data
cursor.execute("DELETE FROM orders;") # Clear existing to avoid duplicate key errors on re-run
cursor.execute("""
INSERT INTO orders VALUES
(1001, 101, 'Rahul', 'Kolkata', 'Laptop', 1, 65000, '2026-09-01'),
(1002, 102, 'Amit', 'Delhi', 'Mouse', 2, 800, '2026-09-02'),
(1003, 101, 'Rahul', 'Kolkata', 'Keyboard', 1, 2500, '2026-09-03'),
(1004, 103, 'Priya', 'Mumbai', 'Monitor', 2, 15000, '2026-09-04'),
(1005, 104, 'Sneha', 'Pune', 'Laptop', 1, 70000, '2026-09-05'),
(1006, 102, 'Amit', 'Delhi', 'Headphones', 2, 3000, '2026-09-06'),
(1007, 101, 'Rahul', 'Kolkata', 'Mouse', 3, 750, '2026-09-07'),
(1008, 103, 'Priya', 'Mumbai', 'Laptop', 1, 68000, '2026-09-08'),
(1009, 104, 'Sneha', 'Pune', 'Keyboard', 2, 2200, '2026-09-09'),
(1010, 102, 'Amit', 'Delhi', 'Monitor', 1, 14000, '2026-09-10');
""")

# 3. Insert Customers Data
cursor.execute("DELETE FROM customers;")
cursor.execute("""
INSERT INTO customers VALUES
(101, 'Rahul', 'Kolkata', '2026-08-01'),
(102, 'Amit', 'Delhi', '2026-08-05'),
(103, 'Priya', 'Mumbai', '2026-08-10'),
(104, 'Sneha', 'Pune', '2026-08-15');
""")

conn.commit()
cursor.close()
conn.close()
print("Tables created and practice data inserted successfully via Jupyter!")

Tables created and practice data inserted successfully via Jupyter!


### ⚙️ Initialize SparkSession for Day 09


In [292]:
import psycopg2
import pandas as pd
from pyspark.sql import SparkSession

# 1. Ensure Spark session is active
if 'spark' not in locals() or not spark:
    spark = SparkSession.builder.appName("Day9_Neon_Pandas_Bridge").getOrCreate()

# 2. Your Neon PostgreSQL Credentials
DB_HOST = "ep-square-frost-b3yvvjy4-pooler.c-4.ap-southeast-1.aws.neon.tech"
DB_NAME = "neondb"
DB_USER = "neondb_owner"
DB_PASSWORD = "npg_yvpcVYCW8F2f"

print("Connecting to Neon PostgreSQL via psycopg2...")

# 3. Connect and query the table using psycopg2 & pandas
conn = psycopg2.connect(
    host=DB_HOST,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    sslmode="require"
)

# Read the 'orders' table directly into a Pandas DataFrame
pdf = pd.read_sql("SELECT * FROM orders;", con=conn)
conn.close()

print(f"Fetched {len(pdf)} rows into Pandas successfully!")

# 4. Convert the Pandas DataFrame into a PySpark DataFrame
orders_pg = spark.createDataFrame(pdf)

# 5. Verify the data in Spark
print(f"Total records read into PySpark: {orders_pg.count()}")
orders_pg.printSchema()
orders_pg.show(5)

Connecting to Neon PostgreSQL via psycopg2...


C:\Users\Me\AppData\Local\Temp\ipykernel_1312\700499037.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pdf = pd.read_sql("SELECT * FROM orders;", con=conn)


Fetched 10 rows into Pandas successfully!
Total records read into PySpark: 10
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- price: double (nullable = true)
 |-- order_date: date (nullable = true)

+--------+-----------+-------------+-------+--------+--------+-------+----------+
|order_id|customer_id|customer_name|   city| product|quantity|  price|order_date|
+--------+-----------+-------------+-------+--------+--------+-------+----------+
|    1001|        101|        Rahul|Kolkata|  Laptop|       1|65000.0|2026-09-01|
|    1002|        102|         Amit|  Delhi|   Mouse|       2|  800.0|2026-09-02|
|    1003|        101|        Rahul|Kolkata|Keyboard|       1| 2500.0|2026-09-03|
|    1004|        103|        Priya| Mumbai| Monitor|       2|15000.0|2026-09-04|
|    1005|        104|      

In [293]:
import pyspark.sql.functions as F

print("--- 1. Running Data Transformations on orders_pg ---")

# Example transformation: Clean/explore schema and calculate metrics
# (Adjust column names based on your actual table schema, e.g., 'total_amount', 'status', 'order_date')

# Show summary stats of the orders DataFrame
orders_pg.describe().show()

# Example Aggregation: Total orders and revenue by order status (if 'status' and 'amount' exist)
# Let's check available columns dynamically to prevent key errors:
columns = orders_pg.columns
print(f"Available columns in orders table: {columns}")

# Safe aggregation based on common column names
if "status" in columns and "total_amount" in columns:
    status_summary = (
        orders_pg.groupBy("status")
        .agg(
            F.count("order_id").alias("order_count"),
            F.sum("total_amount").alias("total_revenue"),
            F.avg("total_amount").alias("avg_order_value")
        )
        .orderBy(F.desc("total_revenue"))
    )
    print("Order Summary by Status:")
    status_summary.show()
else:
    # Fallback basic grouping on the first couple of columns
    print("Performing basic count grouping...")
    orders_pg.groupBy(columns[1] if len(columns) > 1 else columns[0]).count().show(10)

--- 1. Running Data Transformations on orders_pg ---
+-------+------------------+------------------+-------------+-----+----------+------------------+------------------+
|summary|          order_id|       customer_id|customer_name| city|   product|          quantity|             price|
+-------+------------------+------------------+-------------+-----+----------+------------------+------------------+
|  count|                10|                10|           10|   10|        10|                10|                10|
|   mean|            1005.5|             102.3|         NULL| NULL|      NULL|               1.6|           24125.0|
| stddev|3.0276503540974917|1.1595018087284072|         NULL| NULL|      NULL|0.6992058987801011|30497.971244002445|
|    min|              1001|               101|         Amit|Delhi|Headphones|                 1|             750.0|
|    max|              1010|               104|        Sneha| Pune|     Mouse|                 3|           70000.0|
+-------+--

In [294]:
print("--- 2. Exporting processed data ---")

# Option A: Save processed PySpark DataFrame to a local Parquet file for downstream tasks
output_path = "output_orders_processed.parquet"
orders_pg.write.mode("overwrite").parquet(output_path)
print(f"Successfully saved processed orders to Parquet at: {output_path}")

# Option B: If you want to push results back to Postgres using your psycopg2 connection:
# import psycopg2
# conn = psycopg2.connect(host=DB_HOST, database=DB_NAME, user=DB_USER, password=DB_PASSWORD, sslmode="require")
# cursor = conn.cursor()
# # write logic or execute inserts here...
# conn.close()

--- 2. Exporting processed data ---
Successfully saved processed orders to Parquet at: output_orders_processed.parquet


### ⚙️ Initialize SparkSession for Day 09


In [295]:
import psycopg2
import pandas as pd
from pyspark.sql import SparkSession

# 1. Ensure Spark session is active
if 'spark' not in locals() or not spark:
    spark = SparkSession.builder.appName("Day9_Neon_Customer_Revenue").getOrCreate()

# 2. Neon PostgreSQL Credentials
DB_HOST = "ep-square-frost-b3yvvjy4-pooler.c-4.ap-southeast-1.aws.neon.tech"
DB_NAME = "neondb"
DB_USER = "neondb_owner"
DB_PASSWORD = "npg_yvpcVYCW8F2f"

# 3. Define the custom query
query = """
    SELECT 
        customer_id, 
        SUM(quantity * price) AS total_revenue 
    FROM orders 
    GROUP BY customer_id
    ORDER BY total_revenue DESC;
"""

print("Executing query on Neon PostgreSQL...")

# 4. Connect and fetch the query results into a Pandas DataFrame
conn = psycopg2.connect(
    host=DB_HOST,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    sslmode="require"
)

pdf = pd.read_sql(query, con=conn)
conn.close()

print(f"Query executed! Fetched {len(pdf)} rows.")

# 5. Convert the Pandas DataFrame into a PySpark DataFrame
revenue_df = spark.createDataFrame(pdf)

# 6. Show the top customer revenues
revenue_df.show(10, truncate=False)

Executing query on Neon PostgreSQL...


C:\Users\Me\AppData\Local\Temp\ipykernel_1312\2686383354.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pdf = pd.read_sql(query, con=conn)


Query executed! Fetched 4 rows.
+-----------+-------------+
|customer_id|total_revenue|
+-----------+-------------+
|103        |98000.0      |
|104        |74400.0      |
|101        |69750.0      |
|102        |21600.0      |
+-----------+-------------+



#         PostgreSQL
     #         ┌─────────┐
     #         │ orders  │
     #         │customers│
     #         └────┬────┘
     #              │
     #             JDBC
     #              │
     #              ▼
     #          PySpark
     #              │
     #      ┌───────┴────────┐
     #      ▼                ▼
     # Transform           Join
     #      │                │
     #      └───────┬────────┘
     #              ▼
     #          Aggregate
     #              │
     #              ▼
     #         Gold Parquet


---
### 🎓 Day 09 Complete!
Great job completing this module!
👉 **Next Up:** [**Day 10: Custom DAG Pipeline Orchestrator with Retries & Fault Tolerance**](./Day_10_Custom_DAG_Pipeline_Orchestrator.ipynb)
